This notebook includes all the analyses for the association step of the QC pipeline.

Input variables and paths can be set in the designated cells.

This notebook uses several input files:
- final df - the association file after all filtering
- before_prom_df - the association file after filtering for barcode promiscuity
- before_min_assoc_df - the association file after filtering for minimal number of cCRE-barcode associations

All the figures are saved in the output directory that was set



In [ ]:
import const
from const import MPRA_data_paths
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import pysam
import pickle
import regex as re
import collections
import math
const.set_plot_style()
import matplotlib.ticker as ticker
from const import plot_color_pallete
from scipy.optimize import curve_fit
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
import matplotlib.ticker as mtick

In [ ]:
#define paths
project="humanMPRA_L4a1"
#data_path=f"/home/labs/davidgo/Collaboration/MPRA_QC_pipeline/associations/input/{project}"
fasta_path=const.MPRA_data_paths[project]["oligo_fasta"]
working_dir="/home/labs/davidgo/Collaboration/MPRA_QC_pipeline/associations/output/"
output_dir=f"{working_dir}/{project}"
os.makedirs(output_dir,exist_ok=True)

In [ ]:
# define flags
final_flag=1
before_promiscuity_flag=0
before_minimum_associations_flag=0
downsampling_flag=0

In [ ]:
if final_flag:
    final_df=pd.read_csv(const.MPRA_data_paths[project]["association_final"],index_col=0)
if before_promiscuity_flag:
    before_prom_df=pd.read_csv(const.MPRA_data_paths[project]["association_before_promiscuity"],index_col=0)
if before_minimum_associations_flag:
    before_min_assoc_df=pd.read_csv(const.MPRA_data_paths[project]["association_before_minimum_associations"],index_col=0)

In [ ]:
def GC_calc(seq):
    c=seq.count("C")+seq.count("c")
    g=seq.count("G")+seq.count("g")
    perc=(c+g)/len(seq)
    return perc

def counts_df_creator(assoc_df,oligos,f_dict):
    grouped_df=assoc_df.groupby('cCRE')
    assoc_count=grouped_df['match_count'].sum()
    bc_count=grouped_df['barcode'].nunique()
    saved_oligos=grouped_df.groups.keys()
    counts_df=pd.DataFrame(data={"barcode_count":bc_count,"association_count":assoc_count})
    lost_oligos=[oligo for oligo in oligos if oligo not in saved_oligos ]
    add_df=pd.DataFrame(data={"barcode_count":0,"association_count":0},index=lost_oligos)
    full_df=pd.concat([counts_df,add_df])
    full_df['gc']=full_df.index.to_series().apply(lambda x: f_dict[x][0])
    full_df['g_stretch']=full_df.index.to_series().apply(lambda x: f_dict[x][1])
    full_df['len']=full_df.index.to_series().apply(lambda x: f_dict[x][2])
    return full_df

def barcode_df_counts_creator(assoc_df):
    grouped_df=assoc_df.groupby('barcode')
    assoc_count=grouped_df['match_count'].sum()
    oligo_count=grouped_df['cCRE'].nunique()
    counts_df=pd.DataFrame(data={"cCRE_count":oligo_count,"association_count":assoc_count})
    return counts_df

In [ ]:
fasta_file = pysam.FastxFile(fasta_path)
full_oligo_list=set()
feature_dict={}
for entry in fasta_file:
    if entry.name not in full_oligo_list:
        entry_seq=entry.sequence
        full_oligo_list.add(entry.name)
        gc_val=GC_calc(entry_seq)
        g_stretch = len(max(re.findall("[Gg]" + '+', entry_seq), key = len,default=""))
        seq_len=len(entry_seq)
        feature_dict[entry.name]=[gc_val,g_stretch,seq_len]

In [ ]:
if final_flag:
    final_counts_df=counts_df_creator(final_df,full_oligo_list,feature_dict)

### Barcodes per cCRE

In [ ]:
if final_flag:
    plt.clf()
    f, ax_ecdf = plt.subplots()
    sns.ecdfplot(final_counts_df['barcode_count'],color=plot_color_pallete['barcode'],ax=ax_ecdf)

    frac_at_zero = (final_counts_df['barcode_count'] <= 0).mean()   # proportion of values ≤ 0
    frac_at_ten = (final_counts_df['barcode_count'] < 10).mean()
    med = round(final_counts_df['barcode_count'].median())
    avg = final_counts_df['barcode_count'].mean()
    plt.xlabel("Barcode count")
    plt.ylabel("cCREs (%)")
    # Add horizontal line at that y
    plt.axhline(frac_at_zero, color=plot_color_pallete['cCRE'], linestyle="--", linewidth = 2)
    plt.axhline(frac_at_ten, color=plot_color_pallete['cCRE'], linestyle="--", linewidth = 2)
    plt.axvline(med, color=plot_color_pallete['barcode'], linestyle="--", linewidth = 2)
    plt.axvline(avg, color=plot_color_pallete['barcode'], linestyle="--", linewidth = 2)

    plt.text(
        plt.gca().get_xlim()[1]*0.4, frac_at_zero+0.02,  # position text near right side
        f"cCREs with 0 barcodes: {frac_at_zero:.2%}", color=plot_color_pallete['cCRE']
    )
    plt.text(
        plt.gca().get_xlim()[1]*0.4, frac_at_ten+0.05,  # position text near right side
        f"cCREs with fewer than 10 barcodes: {frac_at_ten:.2%}", color=plot_color_pallete['cCRE']
    )
    plt.text(
        med * 2, 0.55,  # position text near right side
        f"Median number of barcodes: {med:d}", color=plot_color_pallete['barcode']
    )
    plt.text(
        avg * 2, 0.65,  # position text near right side
        f"Average number of barcodes: {avg:.0f}", color=plot_color_pallete['barcode']
    )
    ax_ecdf.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f"{val * 100:.0f}%"))

    const.save_fig(plt,"Barcodes per cCRE",output_dir)

 ### cCRE-barcode observations

In [ ]:
if before_minimum_associations_flag:
    plt.clf()
    f, ax_hist = plt.subplots()
    bin_width = 0.5
    min_val = 0
    max_val = 20

    # Define finer bin edges centered on every 0.5 step
    bin_edges = np.arange(min_val - bin_width/2, max_val + bin_width*1.5, bin_width)
    sns.histplot(data=before_min_assoc_df,x='match_count',ax=ax_hist,color=plot_color_pallete["cCRE-barcode-pair"],
                bins=bin_edges,stat='percent')
    ax_hist.set_xlabel("Number of Molecules")
    ax_hist.set_ylabel("cCRE-barcode pairs (%)")
    ax_hist.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax_hist.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}%"))
    ax_hist.set_xticks([1, 20])
    const.save_fig(plt,"cCRE-barcode observations",output_dir)

### Retained cCREs

In [ ]:
if final_flag:
    bc_thresholds=np.arange(0,100,1)
    oligo_counts=[]
    for thr in bc_thresholds:
        pass_sum=final_counts_df['barcode_count'].apply(lambda x: x>thr).sum()
        norm_pass_sum=pass_sum/len(full_oligo_list)
        oligo_counts.append(norm_pass_sum)
    bc_thr_df=pd.DataFrame(data={'threshold':bc_thresholds,'perc':oligo_counts})

In [ ]:
if final_flag:
    fig, ax = plt.subplots()
    sns.lineplot(data=bc_thr_df,x="threshold",y="perc",color=plot_color_pallete['cCRE'],linewidth = 3)
    ax.set_ylabel("cCREs retained (%)")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f"{val * 100:.0f}%"))
    ax.set_xlabel("Minimum barcodes per cCRE")
    ax.set_ylim(0,1)
    const.save_fig(plt,"Retained cCREs",output_dir)

### Barcode promiscuity

In [ ]:
if before_promiscuity_flag:
    promiscuity_counts_df=barcode_df_counts_creator(before_prom_df)

In [ ]:
if before_promiscuity_flag:
    plt.clf()
    f, ax_hist = plt.subplots()
    bin_width = 0.5
    min_val = promiscuity_counts_df['cCRE_count'].min()
    max_val = 10

    # Define finer bin edges centered on every 0.5 step
    bin_edges = np.arange(min_val - bin_width/2, max_val + bin_width*1.5, bin_width)

    sns.histplot(
        data=promiscuity_counts_df,
        x='cCRE_count',
        ax=ax_hist,
        bins=bin_edges,
        color=plot_color_pallete['barcode'],
        edgecolor=None,
        stat='percent'
    )
    ax_hist.set_xlabel(f'Number of cCREs per barcode')
    ax_hist.set_xticks([1, 10])
    ax_hist.set_ylabel("barcodes (%)")
    ax_hist.set_ylim(0, 100)   
    ax_hist.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}%"))
    const.save_fig(plt,"Barcode promiscuity",output_dir)

### PCR bias - GC content

In [ ]:
if final_flag:
    gc_bins=pd.cut(final_counts_df['gc'],bins=np.arange(0,1.01,0.05),duplicates="drop")
    final_counts_df['gc_bin']=gc_bins
    bin_sizes=final_counts_df.reset_index().groupby('gc_bin')['index'].nunique()
    bin_df=pd.DataFrame(data={"gc_bin":bin_sizes.index,"bin_size":bin_sizes.values})

    bin_df['gc_bin_center'] = bin_df['gc_bin'].apply(lambda x: (float(x.left)+float(x.right)))
    bin_intervals = bin_df['gc_bin'].cat.categories
    bin_edges = [i.left for i in bin_intervals] + [bin_intervals[-1].right]
    bin_centers = [(i.left + i.right) / 2 for i in bin_intervals]
    bin_widths = [(i.right - i.left)/2 for i in bin_intervals]

    # Get bar heights in the same order
    heights = bin_df.set_index('gc_bin').loc[bin_intervals]['bin_size'].values
    boxplot_df = final_counts_df.copy()
    boxplot_df['gc_bin_center'] = boxplot_df['gc_bin'].apply(lambda x: (float(x.left)+float(x.right))/2)
    boxplot_groups = boxplot_df.groupby('gc_bin_center')['association_count'].apply(list)

In [ ]:
if final_flag:
    gc_summary = boxplot_df.groupby('gc_bin',observed=False)['association_count'].agg(['count','median']).reset_index()

    f, ax_hist = plt.subplots()

    ax_hist.boxplot(x=boxplot_groups.values,positions=boxplot_groups.index,showfliers=False,widths=bin_widths,
                    patch_artist=True,boxprops=dict(facecolor=plot_color_pallete['read']),
        medianprops=dict(color='black', linewidth=1))
    ax_hist.set_xticks([bin_edges[0],bin_edges[-1]])
    ax_hist.set_xlabel("GC content")
    ax_hist.set_ylabel("Number of reads")
    ax_hist.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax_hist.set_xlim(bin_edges[0], bin_edges[-1])
    ax_hist.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax2 = ax_hist.twinx()
    ax2.plot(boxplot_groups.index, gc_summary['count'],
             color=plot_color_pallete['cCRE'], marker='o', label='cCRE count')
    ax2.set_ylabel("Number of cCREs")
    ax2.yaxis.label.set_color(plot_color_pallete['cCRE'])
    ax_hist.yaxis.label.set_color(plot_color_pallete['read'])
    ax_hist.tick_params(axis='y', colors=plot_color_pallete['read'])
    ax2.tick_params(axis='y', colors=plot_color_pallete['cCRE'])
    ax_hist.spines['right'].set_visible(True)
    f.set_size_inches(8, 8)
    const.save_fig(plt,"PCR bias - GC content",output_dir)

### PCR bias - G-stretches

In [ ]:
if final_flag:
    plt.clf()
    f, ax_hist = plt.subplots()
    g_stretch_summary = final_counts_df.groupby('g_stretch')['association_count'].agg(['count','median']).reset_index()

    sns.boxplot(data=final_counts_df,x="g_stretch",y="association_count",showfliers=False,color=plot_color_pallete['read'],ax=ax_hist)
    #ax_bar.tick_params(axis='x', labelsize=6,rotation=45)
    ax_hist.set_ylabel("Reads per oligo")
    ax_hist.set_xlabel("G stretch")

    ax_hist.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    
    ax2 = ax_hist.twinx()
    ax2.plot(g_stretch_summary['g_stretch'], g_stretch_summary['count'],
             color=plot_color_pallete['cCRE'], marker='o', label='Number of cCREs')
    ax2.set_ylabel("Number of cCREs")
    ax2.yaxis.label.set_color(plot_color_pallete['cCRE'])
    ax_hist.yaxis.label.set_color(plot_color_pallete['read'])
    ax_hist.tick_params(axis='y', colors=plot_color_pallete['read'])
    ax2.tick_params(axis='y', colors=plot_color_pallete['cCRE'])
    ax_hist.spines['right'].set_visible(True)
    f.set_size_inches(8, 8)
    
    const.save_fig(plt,"PCR bias - G-stretches",output_dir)

### Downsampling analysis

In [ ]:
if downsampling_flag:
    data_path=const.MPRA_data_paths[project]["assoc_downsampling"]
    minimum_barcode_sequence_quality = 30
    minimum_number_reads_associating_barcode_to_oligo = 2
    minimum_oligo_mapping_quality = 6
    downsampling_perc_list=np.arange(0.1,1.01,0.1)
    subset="total"

In [ ]:
def downsampling_bc_counts(assoc_df,group):
    if group=='total':
        final_df=assoc_df.copy()
    else:
        final_df=assoc_df[assoc_df['oligo'].apply(lambda x: bool(re.findall(group,x)))].copy() # filter by group - take only necessary oligos
    bc_counts = final_df.groupby('oligo')['bc_seq'].size()
    return pd.DataFrame(data={'bc_counts':bc_counts})

# Hill model
def hill_model(x, a, b, n):
    return a * x**n / (b**n + x**n)
def r_squared(y_true, y_pred):
    ss_total = np.sum((y_true - np.mean(y_true))**2)
    ss_residual = np.sum((y_true - y_pred)**2)
    return 1 - (ss_residual / ss_total)

In [ ]:
if downsampling_flag:
    fasta_file = pysam.FastxFile(fasta_path)
    full_oligo_list=set()
    gc_dict={}
    for entry in fasta_file:
        if entry.name not in full_oligo_list:
            full_oligo_list.add(entry.name)
    total_oligos=len(full_oligo_list)

In [ ]:
if downsampling_flag:
    dfs=[]
    for p in downsampling_perc_list:
        perc=round(p,1)
        print(perc)
        path=fr"{data_path}/oligos_to_barcodes_comb_30_{minimum_number_reads_associating_barcode_to_oligo}_{perc}.txt"
        df=pd.read_csv(path)
        curr_df=downsampling_bc_counts(df, subset)
        curr_df['ds']=perc
        dfs.append(curr_df)

    downsampling_df_total=pd.concat(dfs)
    oligo_coverage=downsampling_df_total.groupby('ds')['bc_counts'].size()/total_oligos
    oligo_coverage_df=pd.DataFrame(data={"ds":oligo_coverage.index.to_list(),"oligo_coverage":oligo_coverage.values})
    oligo_coverage_df['ds']=oligo_coverage_df['ds'].apply(lambda x: str(x))

In [ ]:
if downsampling_flag:
    x_arr=oligo_coverage_df['ds'].to_numpy(dtype=float)
    y_arr=oligo_coverage_df['oligo_coverage'].to_numpy(dtype=float)


    params_hill, _ = curve_fit(hill_model, x_arr, y_arr, bounds=(0, np.inf))

    #create datapoints for plotting
    x_fit = np.linspace(0.1, 3, 100)
    y_hill_fit = hill_model(x_fit,*params_hill)
    r2_hill = r_squared(y_arr, hill_model(x_arr, *params_hill))
    mse_hill = np.mean((y_arr -hill_model(x_arr, *params_hill))**2)
    #predict coverage for higher sequencing values
    x_pred=np.array([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,1.2,1.5,1.75,2,2.5,3])
    y_pred=hill_model(x_pred,*params_hill)
    pred_df=pd.DataFrame(data={"x":x_pred,"predicted coverage_hill":y_pred})
    pred_df.to_csv(fr"{output_dir}/coverage_predictions_hill.csv")


In [ ]:
if downsampling_flag:
    
    sc = plt.scatter(x_pred, y_pred, color='lightgray',marker="s",s=60,label='Hill model fit')
    sc2 = plt.scatter(x_arr, y_arr,color=plot_color_pallete['cCRE'])
    line, = plt.plot(x_fit, y_hill_fit, label='Hill model fit', color='lightgray')
    line2, = plt.plot(x_arr, y_arr, label='Data',color=plot_color_pallete['cCRE'])
    #plt.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax = plt.gca()
    add=y_pred[-6:]-y_arr[-1]
    n = len(x_pred)
    for i in range(n-6, n):
        tic=f"+{add[i+6-n]:.2%}"
        ax.annotate(tic, (x_pred[i], y_pred[i]),
                    xytext=(4, 4), textcoords="offset points",
                    fontsize=9, ha="left", va="bottom")

    hill_proxy = Line2D([0], [0],
                        color='lightgray', linestyle='-',
                        marker='s', markersize=6,
                        markerfacecolor='lightgray', markeredgecolor='lightgray')

    ccre_proxy = Line2D([0], [0],
                        color=plot_color_pallete['cCRE'], linestyle='-',
                        marker='o', markersize=6,
                        markerfacecolor=plot_color_pallete['cCRE'],
                        markeredgecolor=plot_color_pallete['cCRE'])

    plt.legend([hill_proxy, ccre_proxy], ['Model prediction', 'Data'],frameon=False)
    plt.xlabel('Sampling parameter')
    plt.ylabel('Retained cCREs')
    plt.ylim(0,1)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

    const.save_fig(plt,"Downsampling - Retained cCREs",output_dir)

In [ ]:
if downsampling_flag:
    f, ax_box= plt.subplots()
    sns.boxplot(data=downsampling_df_total,y='bc_counts',x='ds',showfliers=False,ax=ax_box,color=plot_color_pallete['barcode'])
    ax_box.set_xlabel("Downsampling parameter")
    ax_box.set_ylabel("Number of Barcodes")
    const.save_fig(plt,"Downsampling - Barcodes per cCRE",output_dir)